# I-PI-MS — Sensitivity Analysis (public code release)

Species-specific sensitivity sweeps and joint geometry optimization. The original optical input CSVs and study-specific VUV calibration factor are not distributed. See `README.md` for local input preparation and reproducibility limitations. This notebook does not require the experimental QMS CSVs.

In [ ]:
from pathlib import Path
import os
from dataclasses import dataclass
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.interpolate import interp1d
from IPython.display import display

DATA_DIR = Path.cwd()     # Directory holding the eight optical input CSVs.
FIGURE_DIR = Path.cwd()   # Figure/CSV outputs on RUN; can point to another directory.
SAVE_OUTPUTS = False  # Opt in to exporting generated figures and tables.
GRID_MM = 0.3             # Keep v22 for direct baseline comparison.
P0_PA = 3.0e-3            # v22 ethanol–acetone run, N2-equivalent reading
CHAMBER_PRESSURE_PA = {'o_xylene': 1.0e-2, 'mixture': P0_PA}  # v22 two-run mapping
T_MIX_K = 353.15
T_XYLENE_K = 298.15
TEMPERATURE_K = [T_XYLENE_K, T_MIX_K, T_MIX_K]
PUMP_NOMINAL_L_S = 520.0
PUMP_ELBOW_CONDUCTANCE_L_S = 12.1 * 30.0**3 / 55.0
PUMP_EFFECTIVE_L_S = 1.0/(1.0/PUMP_NOMINAL_L_S + 1.0/PUMP_ELBOW_CONDUCTANCE_L_S)
GAS_MIN_RADIUS_M = 0.5e-3
PHOTON_PATH_STEP_MM = 0.1
# Local VUV calibration: the study-specific factor is not distributed.
# Set IPI_MS_VUV_SPECTRAL_SCALE in your local environment before starting Jupyter.
# Alternatively assign your own calibrated factor below in a local, untracked copy.
# The factor maps the input spectral ordinate to irradiance [uW/(cm^2 eV)]
# at the reference plane (50 cm from the lamp window; see model geometry).
_scale_text = os.environ.get('IPI_MS_VUV_SPECTRAL_SCALE', '')
try:
    VUV_SPECTRAL_SCALE = float(_scale_text)
except ValueError:
    raise ValueError('Set IPI_MS_VUV_SPECTRAL_SCALE to your own calibrated positive factor.') from None
if not np.isfinite(VUV_SPECTRAL_SCALE) or VUV_SPECTRAL_SCALE <= 0:
    raise ValueError('IPI_MS_VUV_SPECTRAL_SCALE must be finite and positive.')
del _scale_text
XY_PI_OFFSET_MB = 1.5
ACETONE_ABSORPTION_FACTOR = 1.1
REFERENCE_POWER_W = 17  # insert documented measured electrical power to enable watt labels

OPTICAL_FILES = [
    '0-xylene_PI_CS.csv','acetone_PI_CS.csv','ethanol_PI_CS.csv',
    'xylene_abs.csv','acetone_abs.csv','ethanol_abs.csv',
    'Oxyegen_VUV.csv','VUV.csv',
]
PRESSURES_PA = np.array([1e-3,2e-3,3e-3,4e-3,5e-3,7.5e-3,1e-2])
NOZZLE_DISTANCES_MM = np.array([5,5.5,6,7,8,9,10,11,12,12.5,15,18,20,22.5,25],float)
LAMP_WINDOW_DISTANCES_MM = np.array([5,5.5,6,7.5,9,10,11,12,12.5,15.5,19,22.5,25],float)
POWER_RATIOS = np.array([0.1,0.25,0.5,0.75,1.0,1.25,1.5,2.0])
NOZZLE_TILTS_DEG = np.array([-20,-10,-5,0,5,10,20],float)
LAMP_TILTS_DEG = np.array([-8,-4,0,4,8],float)
plt.rcParams.update({
    'figure.dpi': 130, 'savefig.dpi': 300,
    'font.size': 13, 'axes.titlesize': 16, 'axes.labelsize': 14,
    'xtick.labelsize': 12, 'ytick.labelsize': 12,
    'legend.fontsize': 10.5, 'axes.titleweight': 'medium',
    'lines.linewidth': 2.0,
})


## Model geometry and gas supply


In [ ]:
# Exact v22 geometry constants; candidate grid spans the ENTIRE chamber,
# not only the beam-lit baseline: cells can enter/leave the beam during sweeps.
xA, IR_rad = 56.4, 11.0
xB = xA + IR_rad
x_center = (xA + xB)/2.0  # 61.9 mm
R_cyl = 5.0
R_REPELLER_MM = 12.0
h_left, h_right = 7.4, 8.9
x_plane = 49.4          # base VUV window x position (source at x=0)
slope = (h_right-h_left)/(xB-xA)
BASE_LAMP_WINDOW_DISTANCE_MM = x_center-x_plane  # 12.5 mm
BASE_NOZZLE_DISTANCE_MM = 15.0

# Filtering AFTER creating the v22 original 1-D coordinates preserves grid origins.
xs = np.arange(20, xB+10, GRID_MM)
ys = np.arange(-25, 15, GRID_MM)
zs = np.arange(-20, 15, GRID_MM)
xs = xs[(xs>=xA)&(xs<=xB)]
ys = ys[(ys>=-15)&(ys<=10)]
zs = zs[(zs>=-R_cyl)&(zs<=R_cyl)]
XX, YY, ZZ = np.meshgrid(xs,ys,zs,indexing='ij')
within = (XX-x_center)**2 + ZZ**2 <= R_cyl**2
PI_CANDIDATES_MM = np.column_stack([XX[within],YY[within],ZZ[within]])
CELL_VOLUME_CM3 = (GRID_MM*0.1)**3
BASE_BEAM_MASK = (
    (np.hypot(PI_CANDIDATES_MM[:,1],PI_CANDIDATES_MM[:,2])
     <= h_left+slope*(PI_CANDIDATES_MM[:,0]-xA))
    & (PI_CANDIDATES_MM[:,0] >= x_plane)
)
print('v22 geometry: candidate=',len(PI_CANDIDATES_MM),
      ' illuminated=',int(BASE_BEAM_MASK.sum()),
      ' center=',x_center,' window-to-center=',BASE_LAMP_WINDOW_DISTANCE_MM)
assert len(PI_CANDIDATES_MM) == 72210, 'Check GRID_MM and v22 geometry.'
assert BASE_BEAM_MASK.sum() == 44933, 'Baseline beam volume differs from supplied v22/v20.'

In [ ]:
# ---- Source-vial composition and gas-flow boundary conditions ----
MOLAR_MASS_G_MOL = {'o_xylene': 106.165, 'acetone': 58.0791,
                    'ethanol': 46.0684, 'N2': 28.0134, 'O2': 31.9988}
LIQUID_DENSITY_G_ML = {'ethanol': 0.7893, 'acetone': 0.7845}
ANTOINE_NIST = {
    'acetone': (4.42448, 1312.253, -32.445),  # valid 259.16–507.60 K
    'ethanol': (5.24677, 1598.673, -46.424),  # valid 292.77–366.63 K
}

def antoine_pressure_pa(species, temperature_k):
    """Antoine vapor-pressure fit: log10(P/bar) = A - B/(T/K + C)."""
    A, B, C = ANTOINE_NIST[species]
    return 1e5 * 10**(A - B/(temperature_k + C))

# Initial vial inventory assumption: 5 mL ethanol, 5 mL acetone in 50 mL sealed vial.
n_eth_liq = 5.0 * LIQUID_DENSITY_G_ML['ethanol'] / MOLAR_MASS_G_MOL['ethanol']
n_ace_liq = 5.0 * LIQUID_DENSITY_G_ML['acetone'] / MOLAR_MASS_G_MOL['acetone']
x_eth_liq = n_eth_liq/(n_eth_liq+n_ace_liq)
x_ace_liq = n_ace_liq/(n_eth_liq+n_ace_liq)
T_mix = TEMPERATURE_K[1]
p_eth_vial = x_eth_liq * antoine_pressure_pa('ethanol', T_mix)
p_ace_vial = x_ace_liq * antoine_pressure_pa('acetone', T_mix)
p_air_vial = 101325.0 * T_mix/298.15  # Sealed initial air, fixed headspace volume
p_vial_total = p_eth_vial + p_ace_vial + p_air_vial

# o-Xylene at 25 C: 8.8 hPa saturated pressure / 1013 hPa atmospheric pressure.
xyl_fraction = 8.8/1013.0
SOURCE_FRACTIONS = {
    'o_xylene': {'o_xylene': xyl_fraction,
                 'N2': (1-xyl_fraction)*0.79, 'O2': (1-xyl_fraction)*0.21},
    'mixture': {'acetone': p_ace_vial/p_vial_total,
                'ethanol': p_eth_vial/p_vial_total,
                'N2': p_air_vial/p_vial_total*0.79,
                'O2': p_air_vial/p_vial_total*0.21},
}

kB = 1.380649e-23
S_TMP = PUMP_EFFECTIVE_L_S
# Each experimental run has a different indicated chamber pressure.
P_ch_by_condition = dict(CHAMBER_PRESSURE_PA)
if set(P_ch_by_condition) != set(SOURCE_FRACTIONS):
    raise ValueError('Each source condition must have an indicated pressure.')
if any(not np.isfinite(p) or p <= 0 for p in P_ch_by_condition.values()):
    raise ValueError('Chamber pressures must be positive and finite.')
# Inferred throughput Q = S_eff * P_ch (N2-equivalent proxy, not gas-calibrated).
Q_out_by_condition = {
    condition: (S_TMP * 1e-3) * pressure_pa
    for condition, pressure_pa in P_ch_by_condition.items()
}  # Pa m^3 /s

def mean_molecular_speed_m_s(species, temperature_k):
    m_kg = MOLAR_MASS_G_MOL[species] * 1e-3/6.02214076e23
    return np.sqrt(8*kB*temperature_k/(np.pi*m_kg))

def source_field_pa(points_mm, species, condition):
    """Neutral-gas approximation with condition-specific P_ch and Q."""
    p = np.asarray(points_mm, float)
    delta = (p - np.array([x_center, 0.0, -15.0]))*1e-3
    r = np.linalg.norm(delta, axis=-1)
    r_eff = np.maximum(r, GAS_MIN_RADIUS_M)
    cos_theta = np.divide(delta[..., 2], r, out=np.zeros_like(r), where=r>0)
    cos_theta = np.clip(cos_theta, 0.0, 1.0)
    T = TEMPERATURE_K[0] if condition == 'o_xylene' else TEMPERATURE_K[1]
    v = mean_molecular_speed_m_s(species, T)
    frac = SOURCE_FRACTIONS[condition][species]
    P_ch = P_ch_by_condition[condition]
    Q_out = Q_out_by_condition[condition]
    return frac * (P_ch + Q_out*cos_theta/(np.pi*v*r_eff**2))

def species_density_cm3(points_mm, species, condition):
    T = TEMPERATURE_K[0] if condition == 'o_xylene' else TEMPERATURE_K[1]
    return source_field_pa(points_mm, species, condition)/(kB*T)*1e-6

## Ion collection and mesh transmission


In [ ]:
# ---- Finite-thickness QMS mesh transmission ----
@dataclass(frozen=True)
class CylindricalMeshConfig:
    name: str
    radius_mm: float
    thickness_mm: float
    y_min_mm: float
    y_max_mm: float
    hole_axial_mm: float
    hole_circumferential_mm: float
    wire_axial_mm: float
    wire_circumferential_mm: float

    @property
    def pitch_axial_mm(self):
        return self.hole_axial_mm + self.wire_axial_mm

    @property
    def pitch_circumferential_mm(self):
        return self.hole_circumferential_mm + self.wire_circumferential_mm

    @property
    def open_area_fraction(self):
        return (
            self.hole_axial_mm / self.pitch_axial_mm
            * self.hole_circumferential_mm / self.pitch_circumferential_mm
        )


REPELLER_MESH = CylindricalMeshConfig(
    name="QMS electron repeller",
    radius_mm=12.0,
    thickness_mm=0.15,
    y_min_mm=-15.0,
    y_max_mm=15.0,
    hole_axial_mm=0.30,
    hole_circumferential_mm=0.30,
    wire_axial_mm=0.15,
    wire_circumferential_mm=0.15,
)

ION_CHAMBER_MESH = CylindricalMeshConfig(
    name="QMS ionization chamber",
    radius_mm=R_cyl,  # 5 mm outer radius
    thickness_mm=0.30,
    y_min_mm=-15.0,
    y_max_mm=10.0,
    # 1.0 mm (axial) × 2.5 mm (circumferential) clear aperture.
    hole_axial_mm=1.0,
    hole_circumferential_mm=2.5,
    wire_axial_mm=0.30,
    wire_circumferential_mm=0.30,
)

QMS_MESH_OPEN_AREA_TRANSMISSION = (
    REPELLER_MESH.open_area_fraction
    * ION_CHAMBER_MESH.open_area_fraction
)

In [ ]:
def mesh_transmission_for_source(points_mm, mesh, source):
    points=np.asarray(points_mm,float)
    rel=points-source[None,:]
    cx=x_center-source[0]; cz=-source[2]
    px,pz=rel[:,0],rel[:,2]
    A=px*px+pz*pz
    B=-2*(cx*px+cz*pz)
    C=cx*cx+cz*cz-mesh.radius_mm**2
    disc=B*B-4*A*C
    root=np.sqrt(np.maximum(disc,0))
    t1=(-B-root)/(2*A);t2=(-B+root)/(2*A)
    t=np.minimum(np.where((t1>0)&(t1<=1+1e-12),t1,np.inf),
                 np.where((t2>0)&(t2<=1+1e-12),t2,np.inf))
    ok=(disc>=-1e-9)&np.isfinite(t)
    result=np.zeros(len(rel))
    if not ok.any():return result
    pp=rel[ok]; tt=t[ok]; entry=pp*tt[:,None]
    distance=np.linalg.norm(pp,axis=1)
    ray=pp/distance[:,None]
    radial=np.column_stack([entry[:,0]-cx,np.zeros(len(entry)),entry[:,2]-cz])
    radial/=np.linalg.norm(radial,axis=1)[:,None]
    normal=-radial
    tang_ax=np.tile([0.0,1.0,0.0],(len(entry),1))
    tang_circ=np.column_stack([-radial[:,2],np.zeros(len(entry)),radial[:,0]])
    d_n=np.einsum('ij,ij->i',ray,normal)
    d_a=np.einsum('ij,ij->i',ray,tang_ax)
    d_c=np.einsum('ij,ij->i',ray,tang_circ)
    inward=d_n>0
    transmission=np.zeros(len(entry))
    if inward.any():
        depth=np.minimum(mesh.thickness_mm,np.maximum(0,(1-tt[inward])*distance[inward]*d_n[inward]))
        da=depth*d_a[inward]/d_n[inward]
        dc=depth*d_c[inward]/d_n[inward]
        transmission[inward]=(np.clip(mesh.hole_axial_mm-np.abs(da),0,None)/mesh.pitch_axial_mm
                                *np.clip(mesh.hole_circumferential_mm-np.abs(dc),0,None)/mesh.pitch_circumferential_mm)
    result[np.where(ok)[0]]=transmission
    return result

In [ ]:
# Data
r_1 = [44,40,58,40,32,40,44,36,28,60]
r_2 = [22,12,16,16,26,22,10,20,20,20]
r_3 = [8,6,10,4,10,12,12,6,8,6]
r_4 = [4,2,2,2,4,6,4,4,2,10]
r_5 = [2,2,6,2,0,0,0,0,0,6]
r_6 = [2,2,0,0,2,0,6,0,0,0]

a_1 = [58,50,52,54,46,46,58,54,56,46]
a_2 = [10,20,30,18,20,18,20,18,24,20]
a_3 = [8,6,12,8,10,6,8,10,4,16]
a_4 = [4,0,4,0,2,4,0,0,0,4]
a_5 = [2,6,0,2,0,0,0,2,0,2]
a_6 = [0,2,2,0,0,0,0,0,0,0]

e_1 = [56,54,46,48,54,44,48,56,58,58]
e_2 = [18,14,24,22,16,16,24,22,22,28]
e_3 = [4,6,6,6,10,18,8,10,8,8]
e_4 = [4,2,2,2,0,2,2,0,0,0]
e_5 = [2,2,4,4,0,2,0,0,0,0]
e_6 = [0,2,2,2,0,2,0,0,0,0]

groups = [r_1, r_2, r_3, r_4, r_5, r_6]
groups_a = [a_1, a_2, a_3, a_4, a_5, a_6]
groups_e = [e_1, e_2, e_3, e_4, e_5, e_6]


# Bin edges and centers in r/R
edges = np.array([0.0, 0.1, 0.3, 0.5, 0.7, 0.9, 1.0])
centers = (edges[:-1] + edges[1:]) / 2

means = np.array([np.mean(g) for g in groups])
errs  = np.array([np.std(g, ddof=1) for g in groups])  # sample standard deviation
means_a = np.array([np.mean(g) for g in groups_a])
errs_a  = np.array([np.std(g, ddof=1) for g in groups_a])  # sample standard deviation
means_e = np.array([np.mean(g) for g in groups_e])
errs_e  = np.array([np.std(g, ddof=1) for g in groups_e])  # sample standard deviation
# Same v22 nearest-radial-zone SIMION efficiency for ethanol.
normalized_rho = np.hypot(PI_CANDIDATES_MM[:,0]-x_center,
                          PI_CANDIDATES_MM[:,2])/(IR_rad/2.0)
nearest_bin = np.abs(normalized_rho[:,None]-centers[None,:]).argmin(axis=1)
ETA_COLL_ETH = means_e[nearest_bin]*0.01

## VUV photon transport


In [ ]:
missing = [name for name in OPTICAL_FILES if not (DATA_DIR/name).is_file()]
if missing:
    raise FileNotFoundError('Eight optical CSVs required; not embedded in the uploaded notebook:\n'
                            +'\n'.join(missing)+'\nSet DATA_DIR appropriately.')

files = OPTICAL_FILES
names = ['o-Xylene','Acetone','Ethanol','Benzene','Acetone','Ethanol','Oxygen']
refs = ['Zhou et al., 2009','Zhou et al., 2009','Person and Nicole, 1971',
        'Rennie et al., 1998','Nobre et al., 2008',
        'Feng and Brion, 2002','Lu et al., 2010']

energy_range = np.arange(8.0, 10.7, 0.02)

def load_data(file_path):
    data = pd.read_csv(file_path, header=None)
    return data[0].values, data[1].values
def sort_by_energy(energy, cross_section):
    sorted_indices = np.argsort(energy)
    sorted_energy = energy[sorted_indices]
    sorted_cross_section = cross_section[sorted_indices]
    return sorted_energy, sorted_cross_section
def moving_average(data, window_size):
    if window_size < 2:
        return data
    pad_width = window_size // 2
    padded_data = np.pad(data, (pad_width, pad_width), mode='edge')
    smoothed_data = np.convolve(padded_data, np.ones(window_size) / window_size, mode='valid')
    return smoothed_data
def adjust_energy_and_cross_section(energy, cross_section, energy_range):
    min_energy = min(energy)
    additional_energy = np.arange(9.0, min_energy-0.2,0.2)
    additional_cross_section = np.zeros(len(additional_energy))
    adjusted_energy = np.concatenate([additional_energy, energy])
    adjusted_cross_section = np.concatenate([additional_cross_section, cross_section])
    return adjusted_energy, adjusted_cross_section
def extract_from_adjusted(energy, cross_section, energy_range):
    interpolator = interp1d(energy, cross_section, kind='linear', fill_value='extrapolate')
    extracted_cross_section = interpolator(energy_range)
    return extracted_cross_section

energy =[]
cross_section=[]
sorted_energy = []
sorted_cross_section =[]
smooth_cross_section = []
smooth_energy = []
adjusted_energy = []
adjusted_cross_section = []
extracted_cross_section = []

###Samples
for i in range(len(files)-1):
    energy.append(load_data(DATA_DIR / files[i])[0])
    cross_section.append(load_data(DATA_DIR / files[i])[1])
    # Replace negative values in sigma_E with zero
    cross_section[i] = np.where(cross_section[i] < 0, 0, cross_section[i])
    if i == 3: # benzene surrogate for o-xylene
        energy[i], cross_section[i] = sort_by_energy(1240 /energy[i], cross_section[i])
    if i == 4: #acetone
        cross_section[i] = cross_section[i] * ACETONE_ABSORPTION_FACTOR
    if i == 6: #oxygen
        energy[i], cross_section[i] = sort_by_energy(1240 /energy[i], cross_section[i])
    sorted_energy.append(sort_by_energy(energy[i], cross_section[i])[0])
    sorted_cross_section.append(sort_by_energy(energy[i], cross_section[i])[1])
    smooth_cross_section.append(moving_average(sorted_cross_section[i], window_size=3))
    smooth_energy.append(moving_average(sorted_energy[i], window_size=3))
    adjusted_energy.append(adjust_energy_and_cross_section(smooth_energy[i], smooth_cross_section[i], energy_range)[0])
    adjusted_cross_section.append(adjust_energy_and_cross_section(smooth_energy[i], smooth_cross_section[i], energy_range)[1])
    extracted_cross_section.append(extract_from_adjusted(adjusted_energy[i], adjusted_cross_section[i], energy_range))

# VUV lamp spectral data
vuv_data = pd.read_csv(DATA_DIR / files[-1], header=None)
energy_vuv = 1240 /vuv_data[0].values
intensity_vuv = vuv_data[1].values
sorted_energy_vuv, sorted_intensity_vuv = sort_by_energy(energy_vuv, intensity_vuv)
smooth_vuv = moving_average(sorted_intensity_vuv, window_size=3)
smooth_energy_vuv = moving_average(sorted_energy_vuv, window_size=3)
extracted_vuv = extract_from_adjusted(smooth_energy_vuv, smooth_vuv, energy_range)

extracted_vuv = VUV_SPECTRAL_SCALE * extracted_vuv
extracted_cross_section[0] = extracted_cross_section[0] - XY_PI_OFFSET_MB
extracted_cross_section[0][extracted_cross_section[0]<=0] = 0

In [ ]:
# v22 nitrogen Rayleigh surrogate and PI cross-section integration.
h = 6.62607015e-34
c = 299792458.0
kB = 1.380649e-23
Mb_to_cm2 = 1e-18
T_std,P_std = 273.15,101325.0
wavelength_cm = 1240.0/energy_range[::-1]*1e-7
vtilde = 1/wavelength_cm
sigma_um = 1/(wavelength_cm*1e4)
R_std = 1.9662731e6/(22086.66-sigma_um**2)+2.7450825e4/(133.85688-sigma_um**2)
n_std = 1+R_std*1e-6
N_std = P_std/(kB*T_std)*1e-6
F_K = 1.034+3.17e-12*vtilde**2
cross_section_N2 = (24*np.pi**3*vtilde**4/N_std**2
                    *((n_std**2-1)/(n_std**2+2))**2*F_K)
wavelength_nm = 1240.0/energy_range[::-1]
wavelength_m = wavelength_nm*1e-9
I0_lambda = extracted_vuv[::-1]*1e-6/(h*c)*wavelength_m
SIGMA_PI_ETH = extracted_cross_section[2][::-1]*Mb_to_cm2
SIGMA_ABS = {
    'acetone': extracted_cross_section[4][::-1]*Mb_to_cm2,
    'ethanol': extracted_cross_section[5][::-1]*Mb_to_cm2,
    'O2': extracted_cross_section[6][::-1]*Mb_to_cm2,
    'N2': cross_section_N2,
}
assert len(wavelength_nm)==len(SIGMA_PI_ETH)

def local_species_density_cm3(points_mm, species, pressure_pa=P0_PA,
                               nozzle_distance_mm=BASE_NOZZLE_DISTANCE_MM,
                               nozzle_tilt_deg=0.0, condition=None):
    """v22 molecular-flow boundary condition, generalized as in v20 sensitivity notebook."""
    p=np.asarray(points_mm,dtype=float)
    delta=(p-np.array([x_center,0.,-nozzle_distance_mm]))*1e-3
    r=np.linalg.norm(delta,axis=-1)
    aim=np.array([np.sin(np.deg2rad(nozzle_tilt_deg)),0.,np.cos(np.deg2rad(nozzle_tilt_deg))])
    cosine=np.maximum(0.,np.divide(delta@aim,r,out=np.zeros_like(r),where=r>0))
    # o-Xylene uses its independent 298.15-K vial run; acetone/ethanol use the
    # original 353.15-K mixture run. All use the same spatial plume equation.
    condition = condition or ('o_xylene' if species == 'o_xylene' else 'mixture')
    T = T_XYLENE_K if condition == 'o_xylene' else T_MIX_K
    speed = mean_molecular_speed_m_s(species, T)
    frac = SOURCE_FRACTIONS[condition][species]
    p_i = frac*pressure_pa*(1 + PUMP_EFFECTIVE_L_S*1e-3*cosine /
                  (np.pi*speed*np.maximum(r,GAS_MIN_RADIUS_M)**2))
    return p_i/(kB*T)*1e-6

def photon_columns_base_cm2(chunk_size=150):
    """v22 ray integrals: VUV window plane x_plane -> every candidate PI cell."""
    fractions=SOURCE_FRACTIONS['mixture']
    columns={sp:np.empty(len(PI_CANDIDATES_MM)) for sp in fractions}
    for first in range(0,len(PI_CANDIDATES_MM),chunk_size):
        last=min(first+chunk_size,len(PI_CANDIDATES_MM))
        destination=PI_CANDIDATES_MM[first:last]
        origin=destination*(x_plane/destination[:,0])[:,None]
        vec=destination-origin
        path_len_mm=np.linalg.norm(vec,axis=1)
        steps=max(2,int(np.ceil(path_len_mm.max()/PHOTON_PATH_STEP_MM))+1)
        t=np.linspace(0.,1.,steps)
        samples=origin[:,None,:]+t[None,:,None]*vec[:,None,:]
        flat=samples.reshape(-1,3)
        ds_cm=path_len_mm*0.1/(steps-1)
        for sp in fractions:
            density=local_species_density_cm3(flat,sp).reshape(len(destination),steps)
            columns[sp][first:last]=np.trapz(density,axis=1)*ds_cm
    return columns

print('Computing wavelength-dependent optical depth with v22 path integral ...')
BASE_COLUMNS=photon_columns_base_cm2()
BASE_TAU = np.zeros((len(PI_CANDIDATES_MM),len(wavelength_nm)),dtype=np.float32)
for species,col in BASE_COLUMNS.items():
    BASE_TAU += (col[:,None]*SIGMA_ABS[species][None,:]).astype(np.float32)
TAU_BASE_MAX=float(np.nanmax(BASE_TAU))
print('Maximum v22 mixture optical depth:',f'{TAU_BASE_MAX:.3e}')
if TAU_BASE_MAX >= 0.01:
    warnings.warn('Optical depth may be too large to freeze in geometry sweeps; re-integrate rays.')

def unattenuated_gamma_at_pressure(pressure_pa):
    """v22 spectral PI rate/molecule WITHOUT inverse square or meshes; pressure-dependent exp(-tau)."""
    result=np.empty(len(PI_CANDIDATES_MM))
    scale=float(pressure_pa/P0_PA)
    for first in range(0,len(result),400):
        last=min(first+400,len(result))
        phi=(SIGMA_PI_ETH[None,:]*I0_lambda[None,:]
             *np.exp(-BASE_TAU[first:last].astype(np.float64)*scale))
        result[first:last]=np.trapz(phi,wavelength_nm,axis=1)
    return result

GAMMA_BASE=unattenuated_gamma_at_pressure(P0_PA)
GAMMA_CACHE={float(P0_PA):GAMMA_BASE}
def get_gamma(pressure_pa):
    pressure_pa=float(pressure_pa)
    if pressure_pa not in GAMMA_CACHE:
        GAMMA_CACHE[pressure_pa]=unattenuated_gamma_at_pressure(pressure_pa)
    return GAMMA_CACHE[pressure_pa]


## Sensitivity model


In [ ]:
def lamp_ray_factors(window_distance_mm=BASE_LAMP_WINDOW_DISTANCE_MM,
                     lamp_tilt_deg=0.0, remove_repeller=False):
    """Recompute beam overlap, inverse-square irradiance, and finite mesh incidence."""
    # Retracting by Δx means moving lamp virtual source and physical window by -Δx.
    retraction=window_distance_mm-BASE_LAMP_WINDOW_DISTANCE_MM
    source=np.array([-retraction,0.,0.])
    rel=PI_CANDIDATES_MM-source[None,:]
    distance=np.linalg.norm(rel,axis=1)
    angle=np.deg2rad(lamp_tilt_deg)
    axis=np.array([np.cos(angle),np.sin(angle),0.0])
    parallel=rel@axis
    perpendicular=np.linalg.norm(rel-parallel[:,None]*axis[None,:],axis=1)
    # IMPORTANT: no artificial (parallel>=xA) cutoff; this was a v20 issue.
    beam_radius=h_left+slope*(parallel-xA)
    illuminated=(parallel>0.)&(perpendicular<=beam_radius)
    invsq=(549.4/distance)**2  # from v22 source irradiance convention
    trans_ion=mesh_transmission_for_source(PI_CANDIDATES_MM,ION_CHAMBER_MESH,source)
    trans_rep=np.ones(len(distance)) if remove_repeller else mesh_transmission_for_source(
        PI_CANDIDATES_MM,REPELLER_MESH,source)
    return invsq*trans_ion*trans_rep*illuminated, illuminated, trans_ion, trans_rep

BASE_LAMP_CACHE={}
def get_lamp_factor(window_distance_mm, lamp_tilt_deg, remove_repeller):
    key=(float(window_distance_mm),float(lamp_tilt_deg),bool(remove_repeller))
    if key not in BASE_LAMP_CACHE:
        BASE_LAMP_CACHE[key]=lamp_ray_factors(*key)
    return BASE_LAMP_CACHE[key]

BASE_DENSITY_CHAMBER_CM3 = (SOURCE_FRACTIONS['mixture']['ethanol']*P0_PA
                                 /(kB*T_MIX_K)*1e-6)
def physically_feasible(nozzle_distance_mm,window_distance_mm,remove_repeller):
    min_radius=R_cyl if remove_repeller else R_REPELLER_MM
    # Surface crossing forbidden; no additional nozzle/lamp-body envelope known.
    return nozzle_distance_mm>=min_radius and window_distance_mm>=min_radius

def evaluate_ethanol(pressure_pa=P0_PA,
                     nozzle_distance_mm=BASE_NOZZLE_DISTANCE_MM,
                     lamp_window_distance_mm=BASE_LAMP_WINDOW_DISTANCE_MM,
                     nozzle_tilt_deg=0., lamp_tilt_deg=0.,
                     power_ratio=1.,remove_repeller=False,
                     enforce_clearance=True):
    """Single-parameter design evaluator, Kdet referenced to chamber density as in v22 Table 3."""
    if pressure_pa<=0 or power_ratio<0 or nozzle_distance_mm<=0 or lamp_window_distance_mm<=0:
        raise ValueError('Pressure, distances must be > 0; power ratio >= 0.')
    if enforce_clearance and not physically_feasible(nozzle_distance_mm,lamp_window_distance_mm,remove_repeller):
        return dict(feasible=False, rate_ions_s=np.nan, kdet=np.nan,
                    n_chamber_cm3=np.nan, n_pi_mean_cm3=np.nan, illuminated_cells=np.nan)
    light,mask,_,_=get_lamp_factor(lamp_window_distance_mm,lamp_tilt_deg,remove_repeller)
    density=local_species_density_cm3(PI_CANDIDATES_MM,'ethanol',
               pressure_pa=pressure_pa,nozzle_distance_mm=nozzle_distance_mm,
               nozzle_tilt_deg=nozzle_tilt_deg)
    gamma=get_gamma(pressure_pa)
    rate=float(np.sum(density*gamma*light*ETA_COLL_ETH*CELL_VOLUME_CM3*power_ratio))
    n_ch=float(SOURCE_FRACTIONS['mixture']['ethanol']*pressure_pa/(kB*T_MIX_K)*1e-6)
    return dict(feasible=True,rate_ions_s=rate,kdet=rate/n_ch,
                n_chamber_cm3=n_ch,n_pi_mean_cm3=float(density[mask].mean()),
                illuminated_cells=int(mask.sum()))

base_installed=evaluate_ethanol()
base_removed=evaluate_ethanol(remove_repeller=True)
print('Model reference (installed):',base_installed)
print('Model reference (repeller mesh optically removed):',base_removed)
assert np.isfinite(base_installed['kdet']) and base_installed['kdet']>0
assert base_removed['rate_ions_s']>=base_installed['rate_ions_s']
assert np.isclose(base_installed['n_chamber_cm3'],BASE_DENSITY_CHAMBER_CM3)
# v22 executed reference (only meaningful if same eight spectral CSVs are supplied):
V22_CACHED_ETHANOL_MODEL_IONS_S = 2.562847e4
baseline_relative_error=(base_installed['rate_ions_s']/V22_CACHED_ETHANOL_MODEL_IONS_S-1)
print('Deviation from cached v22 ethanol model rate:',f'{baseline_relative_error:+.2%}')
if abs(baseline_relative_error)>0.025:
    warnings.warn('Baseline differs >2.5% from v22 cached result. Check the exact CSV revisions.')

## Sensitivity parameter sweeps


In [ ]:
SWEEP_DEFINITIONS = [
    ('pressure_pa','Chamber pressure','N2-equivalent indicated pressure [Pa]',PRESSURES_PA,P0_PA),
    ('nozzle_distance_mm','Gas-inlet distance','Gas-inlet–PI chamber center [mm]',NOZZLE_DISTANCES_MM,BASE_NOZZLE_DISTANCE_MM),
    ('lamp_window_distance_mm','VUV lamp distance','VUV window–PI chamber center [mm]',LAMP_WINDOW_DISTANCES_MM,BASE_LAMP_WINDOW_DISTANCE_MM),
    ('power_ratio','VUV electrical power','VUV electrical power [W]',POWER_RATIOS,1.0),
    ('nozzle_tilt_deg','Nozzle angle','Nozzle tilt [degree]',NOZZLE_TILTS_DEG,0.),
    ('lamp_tilt_deg','Lamp angle','Lamp tilt [degree]',LAMP_TILTS_DEG,0.),
]
rows=[]
for key,title,xlabel,values,base_value in SWEEP_DEFINITIONS:
    if not np.any(np.isclose(values,base_value)):
        raise RuntimeError(f'Baseline absent from {key} sweep')
    for remove in [False,True]:
        for value in values:
            result=evaluate_ethanol(**{key:float(value)}, remove_repeller=remove)
            rows.append(dict(parameter=key,title=title,setting=float(value),
                             repeller_mesh=('Removed (optical only)' if remove else 'Installed'),
                             remove_repeller=remove,**result))
SWEEP_TABLE=pd.DataFrame(rows)
SWEEP_TABLE['kdet_unit']='(ion-count-equiv/s)/(molecule/cm^3)'
print('Six parameters x two optical designs:',len(SWEEP_TABLE),'rows')
display(SWEEP_TABLE[['parameter','setting','repeller_mesh','feasible','rate_ions_s','kdet']])
assert set(SWEEP_TABLE.parameter)==set(z[0] for z in SWEEP_DEFINITIONS)
# Pressure invariance diagnostic: proportional density cancels, aside from tiny photon self-absorption.
pressure_rows=SWEEP_TABLE[SWEEP_TABLE.parameter=='pressure_pa']
for removed in [False,True]:
    y=pressure_rows[pressure_rows.remove_repeller==removed]['kdet'].to_numpy()
    drift=(np.max(y)-np.min(y))/np.mean(y)
    print(f'Kdet variation across pressure range | remove={removed}: {100*drift:.5f}%')

## Best-fit selection


In [ ]:
def model_matrix(name, x):
    x = np.asarray(x, float)
    if name == 'constant': return np.ones((len(x), 1))
    if name == 'inverse_1': return np.column_stack([np.ones_like(x), 1/x])
    if name == 'inverse_2': return np.column_stack([np.ones_like(x), 1/x**2])
    if name == 'linear': return np.column_stack([np.ones_like(x), x])
    if name == 'quadratic': return np.column_stack([np.ones_like(x), x, x*x])
    if name == 'origin_linear': return x[:, None]
    raise KeyError(name)

FIT_FORMULAS = {
    'constant': r'$c$',
    'inverse_1': r'$a+b/d$',
    'inverse_2': r'$a+b/d^2$',
    'linear': r'$a+bd$',
    'quadratic': r'$a+bd+cd^2$',
    'origin_linear': r'$b(P/P_0)$',
}
FIT_CANDIDATES = {
    'pressure_pa': ['constant'],
    'nozzle_distance_mm': ['inverse_2', 'inverse_1'],
    'lamp_window_distance_mm': ['linear', 'inverse_2', 'quadratic'],
    'power_ratio': ['origin_linear', 'linear'],
}

def fit_curve(x, y, name):
    x = np.asarray(x, float); y = np.asarray(y, float)
    X = model_matrix(name, x)
    coef, *_ = np.linalg.lstsq(X, y, rcond=None)
    pred = X @ coef
    rss = float(np.sum((y-pred)**2))
    tss = float(np.sum((y-y.mean())**2))
    r2 = (np.nan if name == 'constant' or tss < 1e-12*max(np.sum(y*y),1e-300)
          else 1-rss/tss)
    maxerr = float(np.max(np.abs((y-pred)/np.maximum(np.abs(y),1e-300))))
    loo_res = []
    for i in range(len(x)):
        leave = np.ones(len(x), dtype=bool); leave[i] = False
        coef_loo, *_ = np.linalg.lstsq(X[leave], y[leave], rcond=None)
        loo_res.append(float((X[i] @ coef_loo)-y[i]))
    loocv_rmse = float(np.sqrt(np.mean(np.square(loo_res))))
    return dict(coefficients=coef, r2=r2, rmse=np.sqrt(rss/len(y)),
                loocv_rmse=loocv_rmse,
                loocv_relative_pct=100*loocv_rmse/max(float(np.mean(np.abs(y))),1e-300),
                max_relative_residual_pct=100*maxerr)

fits = []
for key, candidates in FIT_CANDIDATES.items():
    for remove in (False, True):
        cur = SWEEP_TABLE[(SWEEP_TABLE.parameter == key)
                          & (SWEEP_TABLE.remove_repeller == remove)
                          & SWEEP_TABLE.feasible].copy()
        x = cur.setting.to_numpy(); y = cur.kdet.to_numpy()
        for model in candidates:
            ft = fit_curve(x, y, model)
            fits.append(dict(parameter=key, remove_repeller=remove, model=model,
                             n_points=len(x), fit_min=x.min(), fit_max=x.max(),
                             coefficients=list(ft['coefficients']), R2=ft['r2'],
                             rmse_kdet=ft['rmse'], loocv_rmse=ft['loocv_rmse'],
                             loocv_relative_pct=ft['loocv_relative_pct'],
                             max_rel_residual_pct=ft['max_relative_residual_pct']))
FIT_TABLE = pd.DataFrame(fits)

# Choose one parsimonious best-fit function for each configuration, on the full feasible range.
BEST_FITS = {}
for key in FIT_CANDIDATES:
    for remove in (False, True):
        sub = FIT_TABLE[(FIT_TABLE.parameter == key)
                        & (FIT_TABLE.remove_repeller == remove)]
        if key == 'pressure_pa':
            chosen = 'constant'
        elif key == 'power_ratio':
            chosen = 'origin_linear'  # exact assumption: optical photons scale linearly with P/P0
        else:
            rank = sub.sort_values('loocv_relative_pct')
            chosen = str(rank.iloc[0].model)
            if key == 'lamp_window_distance_mm' and chosen == 'quadratic':
                simple = sub[sub.model != 'quadratic'].sort_values('loocv_relative_pct').iloc[0]
                quadratic = rank.iloc[0]
                if quadratic.loocv_relative_pct >= 0.95*simple.loocv_relative_pct:
                    chosen = str(simple.model)
        BEST_FITS[(key, remove)] = chosen
FIT_TABLE['selected_for_main'] = [BEST_FITS[(row.parameter, row.remove_repeller)] == row.model
                                   for row in FIT_TABLE.itertuples()]
# Appendix §9 displays only ethanol / repeller-removed diagnostic statistics.


def fit_equation_label(model, coefficients, key):
    """Best-fit equation in actual axis variables; coefficients: 2 significant figures.

    d is distance in mm; q=P_elec/P0 (unitless). K has plotted SI-equivalent units.
    """
    c = np.asarray(coefficients, dtype=float)
    fmt = lambda value: f'{abs(float(value)):.2g}'
    def term(v, factor):
        return (' + ' if v >= 0 else ' - ') + fmt(v) + factor
    if model == 'constant':
        return 'K = ' + f'{c[0]:.2g}'
    if model == 'origin_linear':
        return 'K = ' + f'{c[0]:.2g}' + '(P/P₀)'
    if model in ('inverse_1', 'inverse_2'):
        factor = '/d' if model == 'inverse_1' else '/d²'
        return 'K = ' + f'{c[0]:.2g}' + term(c[1],factor)
    if model == 'linear':
        factor = '(P/P₀)' if key == 'power_ratio' else 'd'
        return 'K = ' + f'{c[0]:.2g}' + term(c[1],factor)
    if model == 'quadratic':
        return 'K = ' + f'{c[0]:.2g}' + term(c[1],'d') + term(c[2],'d²')
    raise KeyError(model)


## Main figure — Sensitivity versus design parameters


In [ ]:
MAIN_KEYS = ['pressure_pa','nozzle_distance_mm','lamp_window_distance_mm','power_ratio']
APPENDIX_KEYS = ['nozzle_tilt_deg','lamp_tilt_deg']


def add_mesh_boundaries(ax, parameter, *, mesh_free_only=False):
    if parameter not in ('nozzle_distance_mm','lamp_window_distance_mm'):
        return
    # Shaded obstruction regions stay visible but intentionally have NO legend entries.
    ax.axvspan(0, R_cyl, color='0.65', alpha=0.17, label='_nolegend_')
    if not mesh_free_only:
        ax.axvspan(R_cyl, R_REPELLER_MM, color='#9b6c4d', alpha=0.08,
                   label='_nolegend_')
    ax.axvline(R_cyl, color='0.40', ls=':', lw=1.5,)
    ax.axvline(R_REPELLER_MM, color='#9b6c4d', ls='--', lw=1.5)
    ax.set_xlim(0, 25)
    ax.set_xticks(np.arange(0, 26, 5))


def plot_sweeps(keys, columns=2, show_best_fits=False):
    nrows = int(np.ceil(len(keys)/columns))
    fig, axes = plt.subplots(nrows, columns, figsize=(16.4,5.7*nrows),
                             constrained_layout=True)
    axes = np.atleast_1d(axes).ravel()
    cfg = {rec[0]:rec for rec in SWEEP_DEFINITIONS}
    for panel_idx, (ax,key) in enumerate(zip(axes,keys)):
        title,xlabel = cfg[key][1:3]
        for remove,color,marker,label in (
            (False,'#245e8f','o','Electron Repeller installed'),
            (True,'#da7a29','s','Electron Repeller removed')):
            g = SWEEP_TABLE[(SWEEP_TABLE.parameter == key)
                            & (SWEEP_TABLE.remove_repeller == remove)
                            & SWEEP_TABLE.feasible].sort_values('setting')
            x = g.setting.to_numpy(); y = g.kdet.to_numpy()
            xscale = (REFERENCE_POWER_W if key=='power_ratio' and
                      REFERENCE_POWER_W is not None else 1.0)
            has_fit = show_best_fits and key in FIT_CANDIDATES and key != 'pressure_pa'
            ax.plot(x*xscale, y, ls='none' if has_fit else '-',
                    marker=marker,color=color,ms=6.0,
                    lw=1.9, label='_nolegend_' if has_fit else label)
            if has_fit:
                chosen = BEST_FITS[(key,remove)]
                fit = fit_curve(x,y,chosen)
                xx = np.linspace(x.min(), min(x.max(),25) if key in
                                 ('nozzle_distance_mm','lamp_window_distance_mm') else x.max(),350)
                yy = model_matrix(chosen,xx) @ fit['coefficients']
                ax.plot(xx*xscale,yy,color=color,lw=2.4,
                        label=label + ' fit:\n' +
                              fit_equation_label(chosen,fit['coefficients'],key))
        base = cfg[key][4]
        if key == 'power_ratio' and REFERENCE_POWER_W is not None:
            base *= REFERENCE_POWER_W
            xlabel = 'Electrical power [W]'
        ax.axvline(base,color='0.6',ls=':',lw=1.1,label='_nolegend_')
        ax.set(title=f"({chr(ord('a') + panel_idx)}) {title}",xlabel=xlabel,
               ylabel=r'$K_{\mathrm{det,lab\_PI}}$ [ions s$^{-1}$ / (molecule cm$^{-3}$)]')
        ax.grid(alpha=0.22)
        ax.ticklabel_format(axis='y',style='sci',scilimits=(0,0))
        if key=='pressure_pa':
            ax.ticklabel_format(axis='x',style='sci',scilimits=(0,0))
        add_mesh_boundaries(ax,key)
        ax.legend(loc='best',fontsize=12,framealpha=0.90)
    for ax in axes[len(keys):]:
        ax.axis('off')
    return fig

main_fig = plot_sweeps(MAIN_KEYS,show_best_fits=True)
if SAVE_OUTPUTS:
    FIGURE_DIR.mkdir(exist_ok=True,parents=True)
    main_fig.savefig(FIGURE_DIR/'IPIMS_Ethanol_Kdet_Main_4params_v26.png',
                     dpi=300,bbox_inches='tight')
plt.show();plt.close(main_fig)


## Appendix figure — Ethanol angular dependence


In [ ]:
appendix_fig = plot_sweeps(APPENDIX_KEYS)
if SAVE_OUTPUTS:
    appendix_fig.savefig(FIGURE_DIR/'IPIMS_Ethanol_Kdet_Appendix_2angles.png',
                         dpi=300,bbox_inches='tight')
plt.show(); plt.close(appendix_fig)


## Appendix — Alternative fits


In [ ]:
# Mesh-free ethanol only: one row with nozzle and lamp candidate models.
fig, axes = plt.subplots(1,2,figsize=(16.0,5.8),constrained_layout=True)
for panel_idx, (ax,key) in enumerate(zip(axes,['nozzle_distance_mm','lamp_window_distance_mm'])):
    cur=SWEEP_TABLE[(SWEEP_TABLE.parameter==key)
                    & (SWEEP_TABLE.remove_repeller==True)
                    & SWEEP_TABLE.feasible].sort_values('setting')
    x,y=cur.setting.to_numpy(),cur.kdet.to_numpy()
    ax.scatter(x,y,s=40,c='black',label='Model samples',zorder=5)
    xx=np.linspace(x.min(),x.max(),350)
    for model in FIT_CANDIDATES[key]:
        f=fit_curve(x,y,model)
        chosen=model==BEST_FITS[(key,True)]
        ax.plot(xx,model_matrix(model,xx)@f['coefficients'],
                lw=2.7 if chosen else 1.5,ls='-' if chosen else '--',
                label=('Selected: ' if chosen else '')+
                      FIT_FORMULAS[model]+f' · LOOCV {f["loocv_relative_pct"]:.2g}%')
    ax.set(title=f"({chr(ord('a') + panel_idx)}) "+('Nozzle' if key=='nozzle_distance_mm' else 'VUV lamp')+
                ' distance · Electron Repeller removed',
           xlabel='Distance from ionization chamber center [mm]',
           ylabel=r'$K_{\mathrm{det,PI}}$')
    add_mesh_boundaries(ax,key,mesh_free_only=True)
    ax.grid(alpha=0.2)
    ax.ticklabel_format(axis='y',style='sci',scilimits=(0,0))
    ax.legend(loc='best',fontsize=10)
if SAVE_OUTPUTS:
    fig.savefig(FIGURE_DIR/'IPIMS_Ethanol_Kdet_Appendix_MeshFree_FitComparison_v26.png',
                dpi=300,bbox_inches='tight')
plt.show();plt.close(fig)

# Only outer-repeller-removed ethanol values are published in this Appendix table.
ETHANOL_APPENDIX_FIT_TABLE=FIT_TABLE[
    FIT_TABLE.remove_repeller & FIT_TABLE.parameter.isin(FIT_CANDIDATES)].copy()
display(ETHANOL_APPENDIX_FIT_TABLE[
    ['parameter','model','n_points','R2','loocv_relative_pct',
     'max_rel_residual_pct','selected_for_main','coefficients']])
if SAVE_OUTPUTS:
    SWEEP_TABLE.to_csv(FIGURE_DIR/'IPIMS_Ethanol_Kdet_Sweep_Data_v26.csv',index=False)
    ETHANOL_APPENDIX_FIT_TABLE.to_csv(
        FIGURE_DIR/'IPIMS_Ethanol_Kdet_Appendix_MeshFree_FitStatistics_v26.csv',index=False)


## Notes


## Appendix — o-Xylene and acetone


In [ ]:
# Source-v22 molecular settings: different gas runs and independent SIMION/PI spectra.
APP_SPECIES=('o_xylene','acetone')
APP_LABELS={'o_xylene':'o-Xylene','acetone':'Acetone'}
APP_CONDITION={'o_xylene':'o_xylene','acetone':'mixture'}
APP_TEMPERATURE_K={'o_xylene':T_XYLENE_K,'acetone':T_MIX_K}
APP_BASE_PRESSURE={'o_xylene':CHAMBER_PRESSURE_PA['o_xylene'],
                   'acetone':CHAMBER_PRESSURE_PA['mixture']}
APP_SIMION_ETA={'o_xylene':means[nearest_bin]*0.01,
                'acetone':means_a[nearest_bin]*0.01}
APP_SIGMA_PI={'o_xylene':extracted_cross_section[0][::-1]*Mb_to_cm2,
              'acetone':extracted_cross_section[1][::-1]*Mb_to_cm2}
assert all(len(APP_SIGMA_PI[sp])==len(wavelength_nm) for sp in APP_SPECIES)
assert all(len(APP_SIMION_ETA[sp])==len(PI_CANDIDATES_MM) for sp in APP_SPECIES)

# Compute o-xylene run's VUV optical depth with its own source conditions.
# The acetone mixture optical depth BASE_TAU is already available from section 3.
def photon_columns_for_condition(condition, pressure_pa, chunk_size=150):
    fractions=SOURCE_FRACTIONS[condition]
    columns={sp:np.empty(len(PI_CANDIDATES_MM),dtype=float) for sp in fractions}
    for first in range(0,len(PI_CANDIDATES_MM),chunk_size):
        last=min(first+chunk_size,len(PI_CANDIDATES_MM))
        destination=PI_CANDIDATES_MM[first:last]
        origin=destination*(x_plane/destination[:,0])[:,None]
        length_mm=np.linalg.norm(destination-origin,axis=1)
        steps=max(2,int(np.ceil(length_mm.max()/PHOTON_PATH_STEP_MM))+1)
        tt=np.linspace(0,1,steps)
        samples=origin[:,None,:]+tt[None,:,None]*(destination-origin)[:,None,:]
        ds_cm=length_mm*0.1/(steps-1)
        for sp in fractions:
            dens=local_species_density_cm3(samples.reshape(-1,3),sp,
                                            pressure_pa=pressure_pa,condition=condition)
            columns[sp][first:last]=np.trapz(dens.reshape(len(destination),steps),axis=1)*ds_cm
    return columns

print('Computing xylene-run VUV optical depth from its separate 298.15-K supply ...')
XYLENE_COLUMNS=photon_columns_for_condition('o_xylene', APP_BASE_PRESSURE['o_xylene'])
XYLENE_BASE_TAU=np.zeros_like(BASE_TAU,dtype=np.float32)
XYLENE_ABS_SIGMA={
    'o_xylene':extracted_cross_section[3][::-1]*Mb_to_cm2,  # v22 benzene surrogate
    'O2':SIGMA_ABS['O2'],
    'N2':SIGMA_ABS['N2'],
}
for sp,col in XYLENE_COLUMNS.items():
    XYLENE_BASE_TAU+=(col[:,None]*XYLENE_ABS_SIGMA[sp][None,:]).astype(np.float32)
print('Max xylene optical depth:',f'{float(XYLENE_BASE_TAU.max()):.3e}')
if float(XYLENE_BASE_TAU.max())>=0.01:
    warnings.warn('Xylene baseline optical depth >=0.01: geometry changes may require ray reintegration.')

APPENDIX_GAMMA_CACHE={}
def appendix_gamma(species,pressure_pa):
    if species not in APP_SPECIES:raise ValueError(species)
    p=float(pressure_pa)
    key=(species,p)
    if key not in APPENDIX_GAMMA_CACHE:
        result=np.empty(len(PI_CANDIDATES_MM))
        tau=XYLENE_BASE_TAU if species=='o_xylene' else BASE_TAU
        sigma=APP_SIGMA_PI[species]
        p_base=APP_BASE_PRESSURE[species]
        for start in range(0,len(result),400):
            end=min(start+400,len(result))
            weight=(sigma[None,:]*I0_lambda[None,:]*
                    np.exp(-tau[start:end].astype(float)*(p/p_base)))
            result[start:end]=np.trapz(weight,wavelength_nm,axis=1)
        APPENDIX_GAMMA_CACHE[key]=result
    return APPENDIX_GAMMA_CACHE[key]


def appendix_evaluate_species(species,**overrides):
    """Condition-specific Kdet in repeller-removed optical geometry."""
    if species not in APP_SPECIES:raise ValueError(species)
    kw=dict(pressure_pa=APP_BASE_PRESSURE[species],
            nozzle_distance_mm=BASE_NOZZLE_DISTANCE_MM,
            lamp_window_distance_mm=BASE_LAMP_WINDOW_DISTANCE_MM,
            nozzle_tilt_deg=0.,lamp_tilt_deg=0.,power_ratio=1.)
    kw.update(overrides)
    pressure=float(kw['pressure_pa']);nozzle=float(kw['nozzle_distance_mm'])
    lamp=float(kw['lamp_window_distance_mm']);power=float(kw['power_ratio'])
    if pressure<=0 or nozzle<=0 or lamp<=0 or power<0:
        raise ValueError('Pressure and distances >0; power >=0 required.')
    if not physically_feasible(nozzle,lamp,True):
        return dict(feasible=False,rate_ions_s=np.nan,kdet=np.nan,
                    n_chamber_cm3=np.nan,n_pi_mean_cm3=np.nan,
                    illuminated_cells=np.nan)
    light,mask,_,_=get_lamp_factor(lamp,float(kw['lamp_tilt_deg']),True)
    density=local_species_density_cm3(
        PI_CANDIDATES_MM,species,pressure_pa=pressure,
        nozzle_distance_mm=nozzle,nozzle_tilt_deg=float(kw['nozzle_tilt_deg']))
    gamma=appendix_gamma(species,pressure)
    rate=float(np.sum(density*gamma*light*APP_SIMION_ETA[species]
                      *CELL_VOLUME_CM3*power))
    condition=APP_CONDITION[species]
    n_ch=SOURCE_FRACTIONS[condition][species]*pressure/(kB*APP_TEMPERATURE_K[species])*1e-6
    return dict(feasible=True,rate_ions_s=rate,kdet=rate/n_ch,
                n_chamber_cm3=float(n_ch),
                n_pi_mean_cm3=float(density[mask].mean()) if mask.any() else np.nan,
                illuminated_cells=int(mask.sum()))

reference_values=[]
for sp in APP_SPECIES:
    r=appendix_evaluate_species(sp)
    assert r['feasible'] and np.isfinite(r['kdet']) and r['kdet']>0
    reference_values.append(dict(species=APP_LABELS[sp],**r))
print('Appendix model baselines, outer repeller optically removed:')
display(pd.DataFrame(reference_values))


### Parameter sweeps


In [ ]:
APP_MAIN_KEYS=['pressure_pa','nozzle_distance_mm','lamp_window_distance_mm','power_ratio']
SWEEP_CONFIG={d[0]:d for d in SWEEP_DEFINITIONS}
appendix_rows=[]
for species in APP_SPECIES:
    for key in APP_MAIN_KEYS:
        _,title,xlabel,values,_=SWEEP_CONFIG[key]
        for value in values:
            result=appendix_evaluate_species(species,**{key:float(value)})
            appendix_rows.append(dict(species=species,parameter=key,title=title,
                                      setting=float(value),remove_repeller=True,**result))
APPENDIX_SPECIES_SWEEPS=pd.DataFrame(appendix_rows)
APPENDIX_SPECIES_SWEEPS['kdet_unit']='(ion-count-equiv/s)/(molecule/cm^3)'
assert len(APPENDIX_SPECIES_SWEEPS)==len(APP_SPECIES)*sum(len(SWEEP_CONFIG[k][3]) for k in APP_MAIN_KEYS)
assert APPENDIX_SPECIES_SWEEPS.feasible.all()
for species in APP_SPECIES:
    cur=APPENDIX_SPECIES_SWEEPS[
        (APPENDIX_SPECIES_SWEEPS.species==species)&
        (APPENDIX_SPECIES_SWEEPS.parameter=='pressure_pa')]
    drift=(cur.kdet.max()-cur.kdet.min())/cur.kdet.mean()*100
    print(f'{APP_LABELS[species]} mesh-free pressure Kdet drift: {drift:.4g}%')
# Data available for review; not displayed as a very long DataFrame in the paper notebook.


### Best-fit selection


In [ ]:
APPENDIX_SPECIES_BEST_FITS={}
for species in APP_SPECIES:
    for key in ('nozzle_distance_mm','lamp_window_distance_mm','power_ratio'):
        cur=APPENDIX_SPECIES_SWEEPS[
            (APPENDIX_SPECIES_SWEEPS.species==species)&
            (APPENDIX_SPECIES_SWEEPS.parameter==key)&
            APPENDIX_SPECIES_SWEEPS.feasible]
        x,y=cur.setting.to_numpy(),cur.kdet.to_numpy()
        candidates=FIT_CANDIDATES[key]
        if key=='power_ratio':
            selected='origin_linear'  # exact power-scaling assumption
        else:
            stats={m:fit_curve(x,y,m) for m in candidates}
            selected=min(stats,key=lambda m:stats[m]['loocv_relative_pct'])
            if key=='lamp_window_distance_mm' and selected=='quadratic':
                simpler=min((m for m in candidates if m!='quadratic'),
                            key=lambda m:stats[m]['loocv_relative_pct'])
                if stats['quadratic']['loocv_relative_pct']>=0.95*stats[simpler]['loocv_relative_pct']:
                    selected=simpler
        APPENDIX_SPECIES_BEST_FITS[(species,key)]=selected


### Appendix figure — Species-dependent sensitivity


In [ ]:
def plot_appendix_species_panel():
    fig,axes=plt.subplots(len(APP_SPECIES),len(APP_MAIN_KEYS),
                         figsize=(22,10.7),constrained_layout=True,squeeze=False)
    for row,species in enumerate(APP_SPECIES):
        for col,key in enumerate(APP_MAIN_KEYS):
            ax=axes[row,col]
            _,title,xlabel,values,baseline=SWEEP_CONFIG[key]
            d=APPENDIX_SPECIES_SWEEPS[
                (APPENDIX_SPECIES_SWEEPS.species==species)&
                (APPENDIX_SPECIES_SWEEPS.parameter==key)].sort_values('setting')
            x,y=d.setting.to_numpy(),d.kdet.to_numpy()
            scale=REFERENCE_POWER_W if key=='power_ratio' and REFERENCE_POWER_W is not None else 1.
            do_fit=key!='pressure_pa'
            ax.plot(x*scale,y,linestyle='none' if do_fit else '-',
                    marker='o',ms=6.1,color='#285b88',lw=2.0,label='3D PI model')
            if do_fit:
                chosen=APPENDIX_SPECIES_BEST_FITS[(species,key)]
                f=fit_curve(x,y,chosen)
                xx=np.linspace(x.min(),min(x.max(),25) if key in
                                 ('nozzle_distance_mm','lamp_window_distance_mm') else x.max(),350)
                yy=model_matrix(chosen,xx)@f['coefficients']
                ax.plot(xx*scale,yy,color='#d36d36',lw=2.5,
                        label='Best fit:\n'+fit_equation_label(chosen,f['coefficients'],key))
            base=APP_BASE_PRESSURE[species] if key=='pressure_pa' else baseline
            ax.axvline(base*scale,color='0.57',lw=1.1,ls=':',label='_nolegend_')
            if key in ('nozzle_distance_mm','lamp_window_distance_mm'):
                add_mesh_boundaries(ax,key,mesh_free_only=True)
            if key=='power_ratio' and REFERENCE_POWER_W is not None:
                xlabel='Electrical power [W]'
            ax.set_title(f"({chr(ord('a') + row*len(APP_MAIN_KEYS) + col)}) "+APP_LABELS[species]+' · '+title,fontsize=15)
            ax.set_xlabel(xlabel,fontsize=13)
            if col==0:
                ax.set_ylabel(r'$K_{\mathrm{det,lab\_PI}}$ [ions s$^{-1}$ / (molecule cm$^{-3}$)]',
                              fontsize=13)
            ax.grid(alpha=0.22)
            ax.ticklabel_format(axis='y',style='sci',scilimits=(0,0))
            if key=='pressure_pa':ax.ticklabel_format(axis='x',style='sci',scilimits=(0,0))
            ax.legend(loc='best',fontsize=9.4,framealpha=0.92)
    return fig

APPENDIX_SPECIES_MAIN_FIG=plot_appendix_species_panel()
if SAVE_OUTPUTS:
    FIGURE_DIR.mkdir(parents=True,exist_ok=True)
    APPENDIX_SPECIES_MAIN_FIG.savefig(
        FIGURE_DIR/'IPIMS_Appendix_Xylene_Acetone_MeshFree_4params_v26.png',
        dpi=300,bbox_inches='tight')
plt.show();plt.close(APPENDIX_SPECIES_MAIN_FIG)
if SAVE_OUTPUTS:
    APPENDIX_SPECIES_SWEEPS.to_csv(
        FIGURE_DIR/'IPIMS_Appendix_Xylene_Acetone_MeshFree_Sweeps_v26.csv',index=False)


### Scope


In [ ]:

# ============================================================
# Additional evaluators: o-Xylene and Acetone
# Reuse v26 Appendix A species-specific physics.
# Supports both original and repeller-removed geometries.
# ============================================================

def evaluate_other_species(
    species,
    pressure_pa=None,
    nozzle_distance_mm=BASE_NOZZLE_DISTANCE_MM,
    lamp_window_distance_mm=BASE_LAMP_WINDOW_DISTANCE_MM,
    nozzle_tilt_deg=0.0,
    lamp_tilt_deg=0.0,
    power_ratio=1.0,
    remove_repeller=False,
    enforce_clearance=True,
):
    if species not in APP_SPECIES:
        raise ValueError(species)

    if pressure_pa is None:
        pressure_pa = APP_BASE_PRESSURE[species]

    if (pressure_pa <= 0 or nozzle_distance_mm <= 0
            or lamp_window_distance_mm <= 0 or power_ratio < 0):
        raise ValueError("Invalid pressure, distance or power.")

    if enforce_clearance and not physically_feasible(
        nozzle_distance_mm,
        lamp_window_distance_mm,
        remove_repeller,
    ):
        return dict(
            feasible=False, rate_ions_s=np.nan,
            kdet=np.nan, n_chamber_cm3=np.nan,
            n_pi_mean_cm3=np.nan, illuminated_cells=np.nan,
        )

    # Optical geometry: original or repeller removed
    light, mask, _, _ = get_lamp_factor(
        lamp_window_distance_mm,
        lamp_tilt_deg,
        remove_repeller,
    )

    # Species-specific neutral-gas density
    density = local_species_density_cm3(
        PI_CANDIDATES_MM,
        species,
        pressure_pa=pressure_pa,
        nozzle_distance_mm=nozzle_distance_mm,
        nozzle_tilt_deg=nozzle_tilt_deg,
        condition=APP_CONDITION[species],
    )

    # Species-specific PI cross section and optical depth
    gamma = appendix_gamma(species, pressure_pa)

    # Species-specific SIMION ion collection efficiency
    rate = float(np.sum(
        density * gamma * light
        * APP_SIMION_ETA[species]
        * CELL_VOLUME_CM3 * power_ratio
    ))

    # Chamber-reference molecular density
    n_ref = (
        SOURCE_FRACTIONS[APP_CONDITION[species]][species]
        * pressure_pa
        / (kB * APP_TEMPERATURE_K[species])
        * 1e-6
    )

    return dict(
        feasible=True,
        rate_ions_s=rate,
        kdet=rate / n_ref,
        n_chamber_cm3=float(n_ref),
        n_pi_mean_cm3=(
            float(density[mask].mean())
            if mask.any() else np.nan
        ),
        illuminated_cells=int(mask.sum()),
    )


def evaluate_oxylene(**kwargs):
    return evaluate_other_species("o_xylene", **kwargs)


def evaluate_acetone(**kwargs):
    return evaluate_other_species("acetone", **kwargs)


# Verify consistency with the existing Appendix model.
for sp in APP_SPECIES:
    original_appendix = appendix_evaluate_species(sp)
    new_evaluator = evaluate_other_species(
        sp, remove_repeller=True
    )
    assert np.isclose(
        original_appendix["kdet"],
        new_evaluator["kdet"],
        rtol=1e-10,
    ), f"Baseline mismatch: {sp}"

print("PASS: o-Xylene and Acetone evaluators ready.")


In [ ]:

import numpy as np
import pandas as pd
from IPython.display import display

# ============================================================
# Joint optimization for three molecular species
# ============================================================

T_INT_S = 0.0311

N0 = BASE_NOZZLE_DISTANCE_MM
L0 = BASE_LAMP_WINDOW_DISTANCE_MM

# Use the actual run-specific reference pressures if updated.
P_REF_BY_SPECIES = {
    "o-Xylene": 1.0e-2,
    "Ethanol": 3.0e-3,
    "Acetone": 3.0e-3,
}

# Each evaluator must use the correct species-specific
# cross sections, molecular mass and gas composition.
EVALUATORS = {
    "o-Xylene": (
        globals().get("evaluate_oxylene")
        or globals().get("evaluate_o_xylene")
    ),
    "Ethanol": globals().get("evaluate_ethanol"),
    "Acetone": globals().get("evaluate_acetone"),
}

missing = [
    sp for sp, fn in EVALUATORS.items()
    if not callable(fn)
]
if missing:
    raise RuntimeError(
        f"Missing species evaluators: {missing}. "
        "Define these using the existing physical model."
    )

designs = {
    "Original": (N0, L0, False),
    "Mesh removed": (N0, L0, True),
    "Mesh + nozzle": (5.0, L0, True),
    "Mesh + lamp": (N0, 5.0, True),
    "Joint optimized": (5.0, 5.0, True),
}

def run_design(species, nozzle_mm, lamp_mm, remove):
    fn = EVALUATORS[species]

    result = fn(
        pressure_pa=P_REF_BY_SPECIES[species],
        nozzle_distance_mm=nozzle_mm,
        lamp_window_distance_mm=lamp_mm,
        nozzle_tilt_deg=0.0,
        lamp_tilt_deg=0.0,
        power_ratio=1.0,
        remove_repeller=remove,
        enforce_clearance=True,
    )

    if not result["feasible"]:
        raise RuntimeError(f"Infeasible geometry: {species}")

    k = result["kdet"]
    if not np.isfinite(k) or k <= 0:
        raise RuntimeError(f"Invalid Kdet: {species}")

    return result

def one_count_density(k):
    return 1.0 / (k * T_INT_S)

detail_rows = []
summary_rows = []

for species in EVALUATORS:

    results = {}

    for name, (nozzle, lamp, remove) in designs.items():
        results[name] = run_design(
            species, nozzle, lamp, remove
        )

    K0 = results["Original"]["kdet"]
    KR = results["Mesh removed"]["kdet"]
    KN = results["Mesh + nozzle"]["kdet"]
    KL = results["Mesh + lamp"]["kdet"]
    KO = results["Joint optimized"]["kdet"]

    gain_mesh = KR / K0
    gain_nozzle = KN / KR
    gain_lamp = KL / KR

    gain_product = (
        gain_mesh * gain_nozzle * gain_lamp
    )
    gain_joint = KO / K0

    for name, (nozzle, lamp, remove) in designs.items():
        r = results[name]

        detail_rows.append({
            "Species": species,
            "Configuration": name,
            "Nozzle [mm]": nozzle,
            "Lamp [mm]": lamp,
            "Mesh removed": remove,
            "Kdet": r["kdet"],
            "Gain": r["kdet"] / K0,
            "n_1count": one_count_density(r["kdet"]),
            "Ion rate": r["rate_ions_s"],
        })

    summary_rows.append({
        "Species": species,
        "K_original": K0,
        "K_optimized": KO,
        "Gain_mesh": gain_mesh,
        "Gain_nozzle": gain_nozzle,
        "Gain_lamp": gain_lamp,
        "Gain_joint": gain_joint,
        "Gain_product": gain_product,
        "Joint/Product": gain_joint / gain_product,
        "n_1count_original": one_count_density(K0),
        "n_1count_optimized": one_count_density(KO),
    })

detail = pd.DataFrame(detail_rows)
summary = pd.DataFrame(summary_rows).set_index("Species")

print("=== Species-dependent optimization ===")

display(summary.style.format({
    "K_original": "{:.3e}",
    "K_optimized": "{:.3e}",
    "Gain_mesh": "{:.2f}",
    "Gain_nozzle": "{:.2f}",
    "Gain_lamp": "{:.2f}",
    "Gain_joint": "{:.2f}",
    "Gain_product": "{:.2f}",
    "Joint/Product": "{:.3f}",
    "n_1count_original": "{:.3e}",
    "n_1count_optimized": "{:.3e}",
}))

print("=== All configurations ===")
display(detail.style.format({
    "Kdet": "{:.3e}",
    "Gain": "{:.2f}",
    "n_1count": "{:.3e}",
    "Ion rate": "{:.3e}",
}))

if SAVE_OUTPUTS:
    FIGURE_DIR.mkdir(parents=True, exist_ok=True)
    summary.to_csv(FIGURE_DIR / "I_PI_MS_Table4_3species.csv")
    detail.to_csv(FIGURE_DIR / "I_PI_MS_joint_optimization_3species.csv", index=False)
    print("CSV files saved.")
